# ClimateDT → EOPF HEALPix Converter

Packages ClimateDT IFS-NEMO output (natively HEALPix level 7) into EOPF-compliant zarr.
Data is retrieved via Polytope — no regridding needed.

**Pipeline**: download GRIB via Polytope → direct zarr packaging → inject STAC

In [ ]:
import logging
from pathlib import Path

from healpix_convert.converters.climatedt import ClimateDTConverter

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")

## Configuration — Surface fields

In [ ]:
DATE = "20200102"
TIME = "0100"
LOCAL_DIR = Path(".")
OUTPUT = LOCAL_DIR / f"S00__ADF_CLMDT_{DATE}T{TIME}.zarr"

converter = ClimateDTConverter(
    date=DATE,
    time=TIME,
    params="134/165/166/167/168",  # sp, u10, v10, t2m, d2m
    levtype="sfc",
    local_dir=LOCAL_DIR,
)

## Step 1 — Prepare (download via Polytope)

In [ ]:
result = converter.prepare(output_path=str(OUTPUT))
print(f"Chunks: {result.n_chunks} | Timesteps: {result.n_times}")

## Step 2 — Convert (all spatial chunks)

In [ ]:
for i in range(result.n_chunks):
    converter.convert_group(i)
    if (i + 1) % 20 == 0:
        print(f"  {i+1}/{result.n_chunks}")

## Step 3 — Consolidate

In [ ]:
converter.consolidate()
print(f"Done: {OUTPUT}")

## Ocean 3D (optional)

Uncomment to also retrieve NEMO ocean temperature on 5 depth levels.

In [ ]:
# converter_ocean = ClimateDTConverter(
#     date      = DATE,
#     time      = "0000",
#     params    = "263501",    # avg_thetao
#     levtype   = "o3d",
#     levelist  = "1/2/3/4/5",
#     local_dir = LOCAL_DIR,
# )
# result_ocean = converter_ocean.prepare(output_path=str(LOCAL_DIR / "S00__ADF_CLDTO_ocean.zarr"))
# for i in range(result_ocean.n_chunks):
#     converter_ocean.convert_group(i)
# converter_ocean.consolidate()

## Validation

In [ ]:
import healpy as hp
import matplotlib.pyplot as plt
import xarray as xr

ds = xr.open_zarr(str(OUTPUT), consolidated=True)
print(ds)

data = ds["t2m"].isel(time=0).values
hp.mollview(
    hp.reorder(data - 273.15, n2r=True),
    flip="geo",
    nest=False,
    cmap="RdBu_r",
    min=-50,
    max=50,
    unit="°C",
    title="ClimateDT T2m — HEALPix level 7",
)
plt.show()